In [ ]:
%%writefile resource_management.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

#define MAX 100
#define FILENAME "resources.txt"
#define REPORTFILE "report.txt"

typedef struct {
    int  id;
    char name[50];
    char category[30];
    char department[30];
    int  quantity;
    int  threshold;
    int  priority;
} Resource;

Resource resources[MAX];
int resCount = 0;

void  printMenu();
void  addResource();
void  updateResource();
void  displayAll(Resource arr[], int n);

void  searchMenu();
int   searchByID(Resource arr[], int n, int id);
int   searchByName(Resource arr[], int n, char *name);
int   searchByCategory(Resource arr[], int n, char *category, int result[]);
int   recursiveBinarySearchByID(Resource arr[], int low, int high, int id);

void  sortMenu();
void  swap(Resource *a, Resource *b);
void  bubbleSortByQuantity(Resource *arr, int n);
void  selectionSortByPriority(Resource *arr, int n);
void  sortByDepartment(Resource *arr, int n);

void  mergeDepartments();
int   isDuplicate(Resource arr[], int n, Resource r);

void  findDuplicates();
void  findDuplicatesRecursive(int index, int *found);

void  analyseResources();
void  displayCriticalResources();

double totalQuantityRecursive(Resource arr[], int n);
void   departmentWiseTotalRecursive(char depts[][30], int deptCount, int index);
void   generateReport();

void  saveToFile();
void  loadFromFile();

int main() {
    loadFromFile();
    int choice;

    do {
        printMenu();
        printf("Enter your choice: ");
        if (scanf("%d", &choice) != 1) {
            printf("Invalid input! Please enter a number.\n");
            while (getchar() != '\n');
            continue;
        }

        switch (choice) {
            case 1:  addResource();               break;
            case 2:  updateResource();             break;
            case 3:  displayAll(resources, resCount); break;
            case 4:  searchMenu();                 break;
            case 5:  sortMenu();                   break;
            case 6:  mergeDepartments();            break;
            case 7:  findDuplicates();              break;
            case 8:  analyseResources();            break;
            case 9:  displayCriticalResources();    break;
            case 10: generateReport();              break;
            case 11: saveToFile();                  break;
            case 12: loadFromFile();                break;
            case 13: printf("Saving data and exiting...\n"); saveToFile(); break;
            default: printf("Invalid choice! Please try again.\n");
        }
        printf("\n");
    } while (choice != 13);

    return 0;
}

void printMenu() {
    printf("\n===== SMART EMERGENCY RESOURCE MANAGEMENT SYSTEM =====\n");
    printf(" 1. Add New Resource\n");
    printf(" 2. Update Resource Details\n");
    printf(" 3. Display All Resources\n");
    printf(" 4. Search Resource (ID / Name / Category)\n");
    printf(" 5. Sort Resources (Quantity / Priority / Department)\n");
    printf(" 6. Merge Resources from Two Departments\n");
    printf(" 7. Identify Duplicate Resources\n");
    printf(" 8. Analyse Resource Availability\n");
    printf(" 9. Display Critical / Low-Stock Resources\n");
    printf("10. Generate Consolidated Report\n");
    printf("11. Save Records to File\n");
    printf("12. Retrieve Records from File\n");
    printf("13. Exit\n");
    printf("========================================================\n");
}

Overwriting resource_management.c


In [ ]:
%%writefile -a resource_management.c

void addResource() {
    if (resCount >= MAX) {
        printf("Resource storage is full!\n");
        return;
    }

    Resource r;
    printf("Enter Resource ID: ");
    scanf("%d", &r.id);

    if (searchByID(resources, resCount, r.id) != -1) {
        printf("Error: a resource with this ID already exists!\n");
        return;
    }

    printf("Enter Resource Name: ");
    scanf(" %49[^\n]", r.name);
    printf("Enter Category (Medicine/Equipment/Bed/Supply): ");
    scanf(" %29[^\n]", r.category);
    printf("Enter Department: ");
    scanf(" %29[^\n]", r.department);
    printf("Enter Quantity Available: ");
    scanf("%d", &r.quantity);
    printf("Enter Minimum Threshold: ");
    scanf("%d", &r.threshold);
    printf("Enter Priority (1-High, 2-Medium, 3-Low): ");
    scanf("%d", &r.priority);

    resources[resCount++] = r;
    printf("Resource added successfully!\n");
}

void updateResource() {
    int id;
    printf("Enter Resource ID to update: ");
    scanf("%d", &id);

    int idx = searchByID(resources, resCount, id);
    if (idx == -1) {
        printf("Resource not found!\n");
        return;
    }

    Resource *r = &resources[idx];
    int choice;
    printf("1.Name 2.Category 3.Department 4.Quantity 5.Threshold 6.Priority\n");
    printf("Choose field to update: ");
    scanf("%d", &choice);

    switch (choice) {
        case 1: printf("New Name: ");        scanf(" %49[^\n]", r->name);       break;
        case 2: printf("New Category: ");    scanf(" %29[^\n]", r->category);   break;
        case 3: printf("New Department: ");  scanf(" %29[^\n]", r->department); break;
        case 4: printf("New Quantity: ");    scanf("%d", &r->quantity);          break;
        case 5: printf("New Threshold: ");   scanf("%d", &r->threshold);         break;
        case 6: printf("New Priority: ");    scanf("%d", &r->priority);          break;
        default: printf("Invalid field choice.\n"); return;
    }
    printf("Resource updated successfully!\n");
}

void displayAll(Resource arr[], int n) {
    if (n <= 0) {
        printf("No resources to display.\n");
        return;
    }
    printf("\n%-5s%-18s%-12s%-12s%-8s%-10s%-8s\n",
           "ID", "Name", "Category", "Department", "Qty", "Thresh", "Prior");
    printf("-----------------------------------------------------------------\n");

    for (int i = 0; i < n; i++) {
        Resource *p = arr + i;
        printf("%-5d%-18s%-12s%-12s%-8d%-10d%-8d\n",
               p->id, p->name, p->category, p->department,
               p->quantity, p->threshold, p->priority);
    }
}

int searchByID(Resource arr[], int n, int id) {
    for (int i = 0; i < n; i++) {
        if (arr[i].id == id) return i;
    }
    return -1;
}

int searchByName(Resource arr[], int n, char *name) {
    for (int i = 0; i < n; i++) {
        if (strcmp(arr[i].name, name) == 0) return i;
    }
    return -1;
}

int searchByCategory(Resource arr[], int n, char *category, int result[]) {
    int cnt = 0;
    for (int i = 0; i < n; i++) {
        if (strcmp(arr[i].category, category) == 0) {
            result[cnt++] = i;
        }
    }
    return cnt;
}

int recursiveBinarySearchByID(Resource arr[], int low, int high, int id) {
    if (low > high) return -1;
    int mid = (low + high) / 2;
    if (arr[mid].id == id) return mid;
    else if (arr[mid].id > id)
        return recursiveBinarySearchByID(arr, low, mid - 1, id);
    else
        return recursiveBinarySearchByID(arr, mid + 1, high, id);
}

void searchMenu() {
    int choice;
    printf("1.By ID  2.By Name  3.By Category\nChoose search type: ");
    scanf("%d", &choice);

    if (choice == 1) {
        int id;
        printf("Enter ID: ");
        scanf("%d", &id);

        Resource temp[MAX];
        memcpy(temp, resources, resCount * sizeof(Resource));
        for (int i = 1; i < resCount; i++) {
            Resource key = temp[i];
            int j = i - 1;
            while (j >= 0 && temp[j].id > key.id) {
                temp[j + 1] = temp[j];
                j--;
            }
            temp[j + 1] = key;
        }
        int idx = recursiveBinarySearchByID(temp, 0, resCount - 1, id);
        if (idx != -1) { printf("Resource Found:\n"); displayAll(&temp[idx], 1); }
        else printf("Resource not found.\n");

    } else if (choice == 2) {
        char name[50];
        printf("Enter Name: ");
        scanf(" %49[^\n]", name);
        int idx = searchByName(resources, resCount, name);
        if (idx != -1) { printf("Resource Found:\n"); displayAll(&resources[idx], 1); }
        else printf("Resource not found.\n");

    } else if (choice == 3) {
        char category[30];
        printf("Enter Category: ");
        scanf(" %29[^\n]", category);
        int result[MAX];
        int cnt = searchByCategory(resources, resCount, category, result);
        if (cnt == 0) { printf("No resources found in this category.\n"); return; }
        printf("Found %d resource(s):\n", cnt);
        for (int i = 0; i < cnt; i++) displayAll(&resources[result[i]], 1);

    } else {
        printf("Invalid choice.\n");
    }
}

Appending to resource_management.c


In [ ]:
%%writefile -a resource_management.c

void swap(Resource *a, Resource *b) {
    Resource temp = *a;
    *a = *b;
    *b = temp;
}

void bubbleSortByQuantity(Resource *arr, int n) {
    for (int i = 0; i < n - 1; i++) {
        for (int j = 0; j < n - i - 1; j++) {
            if ((arr + j)->quantity > (arr + j + 1)->quantity) {
                swap(arr + j, arr + j + 1);
            }
        }
    }
}

void selectionSortByPriority(Resource *arr, int n) {
    for (int i = 0; i < n - 1; i++) {
        int minIdx = i;
        for (int j = i + 1; j < n; j++) {
            if ((arr + j)->priority < (arr + minIdx)->priority) {
                minIdx = j;
            }
        }
        if (minIdx != i) swap(arr + i, arr + minIdx);
    }
}

void sortByDepartment(Resource *arr, int n) {
    for (int i = 0; i < n - 1; i++) {
        for (int j = 0; j < n - i - 1; j++) {
            if (strcmp((arr + j)->department, (arr + j + 1)->department) > 0) {
                swap(arr + j, arr + j + 1);
            }
        }
    }
}

void sortMenu() {
    int choice;
    printf("1.By Quantity  2.By Priority  3.By Department\nChoose sort criteria: ");
    scanf("%d", &choice);

    switch (choice) {
        case 1: bubbleSortByQuantity(resources, resCount);     break;
        case 2: selectionSortByPriority(resources, resCount);  break;
        case 3: sortByDepartment(resources, resCount);         break;
        default: printf("Invalid choice.\n"); return;
    }
    printf("Resources sorted successfully!\n");
    displayAll(resources, resCount);
}

int isDuplicate(Resource arr[], int n, Resource r) {
    if (n <= 0) return 0;
    if (arr[n - 1].id == r.id) return 1;
    return isDuplicate(arr, n - 1, r);
}

void mergeDepartments() {
    char dept1[30], dept2[30];
    printf("Enter first department name: ");
    scanf(" %29[^\n]", dept1);
    printf("Enter second department name: ");
    scanf(" %29[^\n]", dept2);

    Resource merged[MAX];
    int mergedCount = 0;

    for (int i = 0; i < resCount; i++) {
        if (strcmp(resources[i].department, dept1) == 0 ||
            strcmp(resources[i].department, dept2) == 0) {
            if (!isDuplicate(merged, mergedCount, resources[i])) {
                merged[mergedCount++] = resources[i];
            }
        }
    }

    if (mergedCount == 0) {
        printf("No resources found for the given departments.\n");
        return;
    }

    printf("\nMerged Resource List (%s + %s), duplicates removed:\n", dept1, dept2);
    displayAll(merged, mergedCount);
}

void findDuplicatesRecursive(int index, int *found) {
    if (index >= resCount) return;

    for (int j = index + 1; j < resCount; j++) {
        if (resources[index].id == resources[j].id) {
            printf("Duplicate -> ID:%d  Name:%s  (rows %d and %d)\n",
                   resources[index].id, resources[index].name, index, j);
            (*found)++;
        }
    }
    findDuplicatesRecursive(index + 1, found);
}

void findDuplicates() {
    int found = 0;
    findDuplicatesRecursive(0, &found);
    if (found == 0) printf("No duplicate resources found.\n");
}

Appending to resource_management.c


In [ ]:
%%writefile -a resource_management.c

void analyseResources() {
    int adequate = 0, low = 0, critical = 0;

    for (int i = 0; i < resCount; i++) {
        int diff = resources[i].quantity - resources[i].threshold;
        if (diff > 5)      adequate++;
        else if (diff >= 0) low++;
        else                critical++;
    }

    printf("\n--- Resource Availability Analysis ---\n");
    printf("Adequate Resources : %d\n", adequate);
    printf("Low Resources       : %d\n", low);
    printf("Critical Resources  : %d\n", critical);
}

void displayCriticalResources() {
    int any = 0;
    printf("\n--- Critical / Low Stock Resources ---\n");
    for (int i = 0; i < resCount; i++) {
        int diff = resources[i].quantity - resources[i].threshold;
        if (diff < 5) {
            char *status = (diff < 0) ? "CRITICAL" : "LOW";
            printf("ID:%d  Name:%s  Dept:%s  Qty:%d  Threshold:%d  Status:%s\n",
                   resources[i].id, resources[i].name, resources[i].department,
                   resources[i].quantity, resources[i].threshold, status);
            any = 1;
        }
    }
    if (!any) printf("All resources are at adequate levels.\n");
}

double totalQuantityRecursive(Resource arr[], int n) {
    if (n <= 0) return 0;
    return arr[n - 1].quantity + totalQuantityRecursive(arr, n - 1);
}

void departmentWiseTotalRecursive(char depts[][30], int deptCount, int index) {
    if (index >= deptCount) return;

    int total = 0;
    for (int i = 0; i < resCount; i++) {
        if (strcmp(resources[i].department, depts[index]) == 0) {
            total += resources[i].quantity;
        }
    }
    printf("Department: %-15s Total Quantity: %d\n", depts[index], total);
    departmentWiseTotalRecursive(depts, deptCount, index + 1);
}

void generateReport() {
    FILE *fp = fopen(REPORTFILE, "w");
    if (fp == NULL) {
        printf("Error creating report file.\n");
        return;
    }

    printf("\n========= CONSOLIDATED RESOURCE REPORT =========\n");
    fprintf(fp, "========= CONSOLIDATED RESOURCE REPORT =========\n");

    double total = totalQuantityRecursive(resources, resCount);
    printf("Total Resources Available (all departments): %.0f\n", total);
    fprintf(fp, "Total Resources Available (all departments): %.0f\n", total);

    char depts[MAX][30];
    int deptCount = 0;
    for (int i = 0; i < resCount; i++) {
        int exists = 0;
        for (int j = 0; j < deptCount; j++) {
            if (strcmp(depts[j], resources[i].department) == 0) { exists = 1; break; }
        }
        if (!exists) strcpy(depts[deptCount++], resources[i].department);
    }

    printf("\nDepartment-wise Availability:\n");
    fprintf(fp, "\nDepartment-wise Availability:\n");
    departmentWiseTotalRecursive(depts, deptCount, 0);
    for (int i = 0; i < deptCount; i++) {
        int deptTotal = 0;
        for (int j = 0; j < resCount; j++) {
            if (strcmp(resources[j].department, depts[i]) == 0) deptTotal += resources[j].quantity;
        }
        fprintf(fp, "Department: %-15s Total Quantity: %d\n", depts[i], deptTotal);
    }

    printf("\nMost Critical Resources (immediate replenishment needed):\n");
    fprintf(fp, "\nMost Critical Resources (immediate replenishment needed):\n");
    int any = 0;
    for (int i = 0; i < resCount; i++) {
        if (resources[i].quantity - resources[i].threshold < 0) {
            printf("ID:%d Name:%s Dept:%s Qty:%d (Needed:%d)\n",
                   resources[i].id, resources[i].name, resources[i].department,
                   resources[i].quantity, resources[i].threshold);
            fprintf(fp, "ID:%d Name:%s Dept:%s Qty:%d (Needed:%d)\n",
                    resources[i].id, resources[i].name, resources[i].department,
                    resources[i].quantity, resources[i].threshold);
            any = 1;
        }
    }
    if (!any) {
        printf("No critical resources at the moment.\n");
        fprintf(fp, "No critical resources at the moment.\n");
    }

    fclose(fp);
    printf("\nReport generated and saved to %s\n", REPORTFILE);
}

void saveToFile() {
    FILE *fp = fopen(FILENAME, "w");
    if (fp == NULL) {
        printf("Error opening file for writing.\n");
        return;
    }
    for (int i = 0; i < resCount; i++) {
        fprintf(fp, "%d|%s|%s|%s|%d|%d|%d\n",
                resources[i].id, resources[i].name, resources[i].category,
                resources[i].department, resources[i].quantity,
                resources[i].threshold, resources[i].priority);
    }
    fclose(fp);
    printf("Records saved to file successfully!\n");
}

void loadFromFile() {
    FILE *fp = fopen(FILENAME, "r");
    if (fp == NULL) {
        return;
    }

    resCount = 0;
    while (resCount < MAX &&
           fscanf(fp, "%d|%49[^|]|%29[^|]|%29[^|]|%d|%d|%d\n",
                  &resources[resCount].id, resources[resCount].name,
                  resources[resCount].category, resources[resCount].department,
                  &resources[resCount].quantity, &resources[resCount].threshold,
                  &resources[resCount].priority) == 7) {
        resCount++;
    }
    fclose(fp);
    printf("Records loaded from file successfully! (%d records)\n", resCount);
}

Appending to resource_management.c


In [ ]:
!gcc -Wall -o resmgmt resource_management.c && ./resmgmt


===== SMART EMERGENCY RESOURCE MANAGEMENT SYSTEM =====
 1. Add New Resource
 2. Update Resource Details
 3. Display All Resources
 4. Search Resource (ID / Name / Category)
 5. Sort Resources (Quantity / Priority / Department)
 6. Merge Resources from Two Departments
 7. Identify Duplicate Resources
 8. Analyse Resource Availability
 9. Display Critical / Low-Stock Resources
10. Generate Consolidated Report
11. Save Records to File
12. Retrieve Records from File
13. Exit
Enter your choice: 1
Enter Resource ID: 101
Enter Resource Name: Paracetamol
Enter Category (Medicine/Equipment/Bed/Supply): Medicine
Enter Department: Emergency
Enter Quantity Available: 20
Enter Minimum Threshold: 30
Enter Priority (1-High, 2-Medium, 3-Low): 1
Resource added successfully!


===== SMART EMERGENCY RESOURCE MANAGEMENT SYSTEM =====
 1. Add New Resource
 2. Update Resource Details
 3. Display All Resources
 4. Search Resource (ID / Name / Category)
 5. Sort Resources (Quantity / Priority / Department)
 6